# Problem Set 11: Project Genesis – The Scholar-Prime (Week 11)

Welcome to Week 11. In this assignment, you will build an academic research agent named **Scholar-Prime** that queries scientific databases to automatically find relevant research papers for simulation modeling. You will use the open-source **Science-Skills** repository developed by **Google DeepMind**.

## Objectives:
1. **Setting up the Science Skills** (Cloning and Environment Setup)
2. **Building the Literature Retrieval Agent (`agent.py`)**
3. **Automated Search & Downloader (Tool Binding)**
4. **Parameter Extraction & Verification**

Let's build!

## Exercise 1: Setting up the Science Skills

Clone the official Google DeepMind science-skills repository into your workspace, sync the environment dependencies via `uv`, and run a test query on the OpenAlex CLI to resolve a researcher's identity (e.g. "Geoffrey Hinton").

```bash
git clone https://github.com/google-deepmind/science-skills.git
cd science-skills/skills/literature-search-openalex
uv sync
uv run scripts/openalex_cli.py resolve authors "Geoffrey Hinton"
```

Execute the Python validation cell below to verify that the CLI utility exists and responds correctly.

In [6]:
from pathlib import Path
import subprocess


def find_project_root() -> Path:
    """
    Finds the genesis-oracle project directory independently of whether
    the notebook kernel starts in the project root or in Notebooks/.
    """
    current_directory = Path.cwd().resolve()

    for candidate in [current_directory, *current_directory.parents]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "science-skills").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "The genesis-oracle project directory could not be found."
    )


PROJECT_ROOT = find_project_root()

OPENALEX_DIRECTORY = (
    PROJECT_ROOT
    / "science-skills"
    / "skills"
    / "literature_search_openalex"
)

OPENALEX_SCRIPT = (
    OPENALEX_DIRECTORY
    / "scripts"
    / "openalex_cli.py"
)

print("=== EXERCISE 1: SCIENCE-SKILLS VALIDATION ===")
print(f"Project root: {PROJECT_ROOT}")
print(f"OpenAlex directory: {OPENALEX_DIRECTORY}")

if not OPENALEX_DIRECTORY.exists():
    raise FileNotFoundError(
        f"OpenAlex skill directory not found: {OPENALEX_DIRECTORY}"
    )

if not OPENALEX_SCRIPT.exists():
    raise FileNotFoundError(
        f"OpenAlex CLI script not found: {OPENALEX_SCRIPT}"
    )

command = [
    "uv",
    "run",
    "scripts/openalex_cli.py",
    "resolve",
    "authors",
    "Geoffrey Hinton",
]

result = subprocess.run(
    command,
    cwd=OPENALEX_DIRECTORY,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "The OpenAlex author resolution command failed."
    )

print("\n=== OPENALEX AUTHOR RESOLUTION ===")
print(result.stdout)

if "A5108093963" not in result.stdout:
    print(
        "Warning: The previously resolved OpenAlex ID "
        "A5108093963 was not found in the current output."
    )
else:
    print("✓ Geoffrey E. Hinton was resolved successfully.")
    print("✓ OpenAlex author ID: A5108093963")

=== EXERCISE 1: SCIENCE-SKILLS VALIDATION ===
Project root: C:\Users\nilss\Documents\genesis-oracle
OpenAlex directory: C:\Users\nilss\Documents\genesis-oracle\science-skills\skills\literature_search_openalex

=== OPENALEX AUTHOR RESOLUTION ===
[
  {
    "id": "https://openalex.org/A5108093963",
    "display_name": "Geoffrey E. Hinton",
    "hint": 385
  },
  {
    "id": "https://openalex.org/A5110248343",
    "display_name": "Geoffrey E. Hinton",
    "hint": 36
  },
  {
    "id": "https://openalex.org/A5000300454",
    "display_name": "Saurabh Saxena",
    "hint": 63
  },
  {
    "id": "https://openalex.org/A5002428732",
    "display_name": "Geoffrey F. Hinton",
    "hint": 2
  },
  {
    "id": "https://openalex.org/A5098035523",
    "display_name": "James A. Anderson and Geoffrey E. Hinton",
    "hint": 1
  }
]

✓ Geoffrey E. Hinton was resolved successfully.
✓ OpenAlex author ID: A5108093963


## Exercise 2: Building the Literature Retrieval Agent

Define your `scholar_prime` agent inside `agent.py` using the ADK SDK.

Complete the starter template below to match your configured `agent.py`.

In [7]:
import sys
from IPython.display import Markdown, display


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from scholar_prime.agent import root_agent


AGENT_FILE = (
    PROJECT_ROOT
    / "scholar_prime"
    / "agent.py"
)

if not AGENT_FILE.exists():
    raise FileNotFoundError(
        f"Scholar-Prime agent file not found: {AGENT_FILE}"
    )

agent_source = AGENT_FILE.read_text(
    encoding="utf-8"
)

print("=== EXERCISE 2: SCHOLAR-PRIME CONFIGURATION ===")
print(f"Model: {root_agent.model}")
print(f"Name: {root_agent.name}")
print(f"Description: {root_agent.description}")
print(f"Registered tools: {root_agent.tools}")

assert root_agent.name == "scholar_prime"
assert root_agent.model == "gemini-3.5-flash"

print("\n✓ Scholar-Prime was loaded successfully.")

display(
    Markdown(
        "### Complete `scholar_prime/agent.py`\n\n"
        "```python\n"
        f"{agent_source}\n"
        "```"
    )
)

=== EXERCISE 2: SCHOLAR-PRIME CONFIGURATION ===
Model: gemini-3.5-flash
Name: scholar_prime
Description: An academic research agent specialized in querying scientific databases and extracting material parameters.
Registered tools: [<function search_arxiv at 0x0000021DC7E65D00>, <function extract_parameters_from_text at 0x0000021DC7E64FE0>]

✓ Scholar-Prime was loaded successfully.


### Complete `scholar_prime/agent.py`

```python
from pathlib import Path
import subprocess
import re

from google.adk.agents.llm_agent import Agent
from google.genai import types


# Absoluter Pfad zum Hauptverzeichnis von genesis-oracle
PROJECT_ROOT = Path(__file__).resolve().parent.parent

# Pfad zum geklonten arXiv-Skill
ARXIV_SKILL_DIRECTORY = (
    PROJECT_ROOT
    / "science-skills"
    / "skills"
    / "literature_search_arxiv"
)


def search_arxiv(query: str, max_results: int = 5) -> str:
    """
    Searches arXiv for scientific papers by using the Google DeepMind
    Science-Skills command-line tool.

    Args:
        query: The scientific search query sent to arXiv.
        max_results: The maximum number of papers to return.

    Returns:
        The JSON-formatted output produced by the arXiv search tool.
        If the search cannot be completed, an error message is returned.
    """

    if not query.strip():
        return "Error: The arXiv search query must not be empty."

    if max_results < 1:
        return "Error: max_results must be at least 1."

    # Begrenzung, damit keine unnötig große Tool-Antwort entsteht
    max_results = min(max_results, 3)

    script_path = (
        ARXIV_SKILL_DIRECTORY
        / "scripts"
        / "search_arxiv.py"
    )

    if not script_path.exists():
        return (
            "Error: The arXiv search script was not found at "
            f"{script_path}"
        )

    command = [
        "uv",
        "run",
        "scripts/search_arxiv.py",
        "--query",
        query,
        "--max_results",
        str(max_results),
    ]

    try:
        result = subprocess.run(
            command,
            cwd=ARXIV_SKILL_DIRECTORY,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=120,
            check=False,
        )

    except subprocess.TimeoutExpired:
        return (
            "Error: The arXiv search timed out after "
            "120 seconds."
        )

    except OSError as error:
        return (
            "Error: The arXiv search process could not "
            f"be started: {error}"
        )

    if result.returncode != 0:
        error_message = (
            result.stderr.strip()
            or result.stdout.strip()
            or "Unknown command-line error."
        )

        return (
            "Error: The arXiv search command failed.\n"
            f"Details: {error_message}"
        )

    output = result.stdout.strip()

    if not output:
        return "Error: The arXiv search returned no output."

    return output

def extract_parameters_from_text(text: str) -> dict[str, object]:
    """
    Extracts physical and simulation-related parameters from scientific text.

    The function searches for common reactor and material parameters, such as
    thermal conductivity, temperature, density, heat capacity, friction
    coefficients, fission barrier heights, and fission isomer energies.

    Args:
        text: Scientific abstract or paper text from which parameters
            should be extracted.

    Returns:
        A dictionary containing detected parameters, numerical values,
        units, evidence snippets, and an extraction summary.
    """

    if not text or not text.strip():
        return {
            "status": "error",
            "message": "The input text must not be empty.",
            "parameters": [],
        }

    parameter_definitions = [
        {
            "name": "thermal_conductivity",
            "label_pattern": r"thermal conductivity",
            "unit_pattern": r"W\s*/?\s*\(?m\s*[·*]?\s*K\)?",
            "default_unit": "W/(m·K)",
        },
        {
            "name": "temperature",
            "label_pattern": r"(?:temperature|core temperature)",
            "unit_pattern": r"(?:K|°C|Celsius|Kelvin)",
            "default_unit": None,
        },
        {
            "name": "density",
            "label_pattern": (
                r"\b(?:material density|mass density|bulk density|fuel density)\b"
            ),
            "unit_pattern": (
                r"(?:kg\s*/\s*m(?:\^?3|³)|g\s*/\s*cm(?:\^?3|³))"
            ),
            "default_unit": None,
        },
        {
            "name": "specific_heat_capacity",
            "label_pattern": r"(?:specific heat capacity|heat capacity)",
            "unit_pattern": r"J\s*/\s*\(?kg\s*[·*]?\s*K\)?",
            "default_unit": "J/(kg·K)",
        },
        {
            "name": "thermal_friction_coefficient",
            "label_pattern": r"(?:thermal friction coefficient|friction coefficient)",
            "unit_pattern": r"(?:dimensionless|1)",
            "default_unit": "dimensionless",
        },
        {
            "name": "fission_barrier_height",
            "label_pattern": r"(?:inner|outer|fission)?\s*barrier heights?",
            "unit_pattern": r"(?:MeV|keV|eV)",
            "default_unit": "MeV",
        },
        {
            "name": "fission_isomer_energy",
            "label_pattern": r"(?:fission )?isomer energies?",
            "unit_pattern": r"(?:MeV|keV|eV)",
            "default_unit": "MeV",
        },
    ]

    extracted_parameters: list[dict[str, object]] = []

    number_pattern = r"[-+]?\d+(?:[.,]\d+)?(?:[eE][-+]?\d+)?"

    for definition in parameter_definitions:
        label_pattern = definition["label_pattern"]
        unit_pattern = definition["unit_pattern"]

        mention_match = re.search(
            label_pattern,
            text,
            flags=re.IGNORECASE,
        )

        if mention_match is None:
            continue

        value_pattern = (
            rf"{label_pattern}"
            rf".{{0,100}}?"
            rf"(?P<value>{number_pattern})"
            rf"\s*(?P<unit>{unit_pattern})"
        )

        value_match = re.search(
            value_pattern,
            text,
            flags=re.IGNORECASE | re.DOTALL,
        )

        snippet_start = max(0, mention_match.start() - 60)
        snippet_end = min(len(text), mention_match.end() + 120)

        evidence = " ".join(
            text[snippet_start:snippet_end].split()
        )

        if value_match:
            raw_value = value_match.group("value").replace(",", ".")
            unit = value_match.group("unit")

            try:
                value: float | None = float(raw_value)
            except ValueError:
                value = None

            status = "numeric_value_extracted"
        else:
            value = None
            unit = definition["default_unit"]
            status = "mentioned_without_numeric_value"

        extracted_parameters.append(
            {
                "name": definition["name"],
                "value": value,
                "unit": unit,
                "status": status,
                "evidence": evidence,
            }
        )

    formulas = re.findall(
        r"\b[A-Za-z][A-Za-z0-9_]*\s*=\s*[^,.;\n]+",
        text,
    )

    return {
        "status": "success",
        "parameters_found": len(extracted_parameters),
        "parameters": extracted_parameters,
        "formulas": formulas,
        "note": (
            "Parameters without explicit numerical values are stored "
            "with value null instead of being invented."
        ),
    }

root_agent = Agent(
    model="gemini-3.5-flash",

    # Automatische Wiederholungsversuche bei vorübergehenden
    # Netzwerk-, Kapazitäts- und Serverfehlern
    generate_content_config=types.GenerateContentConfig(
        http_options=types.HttpOptions(
            retry_options=types.HttpRetryOptions(
                initial_delay=2.0,
                attempts=8,
                exp_base=2.0,
                max_delay=30.0,
                http_status_codes=[
                    408,
                    429,
                    500,
                    502,
                    503,
                    504,
                ],
            ),
            timeout=120_000,
        )
    ),

    name="scholar_prime",

    description=(
        "An academic research agent specialized in querying "
        "scientific databases and extracting material parameters."
    ),

    instruction=(
        "You are Scholar-Prime, a professional academic research "
        "agent. Use the search_arxiv tool whenever the user requests "
        "scientific literature from arXiv. Evaluate the relevance of "
        "the returned abstracts, identify the most relevant paper, "
        "and summarize its scientific contribution accurately. "
        "Extract important formulas and material parameters when "
        "they are present. Always state the DOI of every cited paper "
        "when a DOI is available. Never invent a DOI, formula, "
        "parameter, author, or paper result. Clearly distinguish "
        "information taken directly from a paper from your own "
        "interpretation. Call search_arxiv only once per user request "
        "unless the tool returns an error or no papers. Request at "
        "most three results and do not repeat the search merely to "
        "refine the wording."
    ),

    tools=[
    search_arxiv,
    extract_parameters_from_text,
    ],
)
```

## Exercise 3: Automated Search & Downloader (Tool Binding)

Implement a Python function `search_arxiv(query: str, max_results: int = 5) -> str` that invokes the arXiv CLI tool from the science-skills repository under the hood. Bind this function to your agent and test it in the Web UI.

Below is the template for the wrapper tool.

In [8]:
import inspect

from scholar_prime.agent import search_arxiv


print("=== EXERCISE 3: ARXIV TOOL BINDING ===")

search_arxiv_source = inspect.getsource(
    search_arxiv
)

display(
    Markdown(
        "### `search_arxiv` tool implementation\n\n"
        "```python\n"
        f"{search_arxiv_source}\n"
        "```"
    )
)

search_query = (
    "thermodynamic simulation parameters "
    "for advanced fission reactors"
)

print(f"Query: {search_query}")
print("Maximum results: 3")
print("\nSearching arXiv...")

arxiv_output = search_arxiv(
    query=search_query,
    max_results=3,
)

if arxiv_output.startswith("Error:"):
    raise RuntimeError(arxiv_output)

print("\n=== ARXIV TOOL OUTPUT ===")
print(arxiv_output)

assert '"status": "success"' in arxiv_output
assert '"papers"' in arxiv_output

print("\n✓ search_arxiv completed successfully.")

=== EXERCISE 3: ARXIV TOOL BINDING ===


### `search_arxiv` tool implementation

```python
def search_arxiv(query: str, max_results: int = 5) -> str:
    """
    Searches arXiv for scientific papers by using the Google DeepMind
    Science-Skills command-line tool.

    Args:
        query: The scientific search query sent to arXiv.
        max_results: The maximum number of papers to return.

    Returns:
        The JSON-formatted output produced by the arXiv search tool.
        If the search cannot be completed, an error message is returned.
    """

    if not query.strip():
        return "Error: The arXiv search query must not be empty."

    if max_results < 1:
        return "Error: max_results must be at least 1."

    # Begrenzung, damit keine unnötig große Tool-Antwort entsteht
    max_results = min(max_results, 3)

    script_path = (
        ARXIV_SKILL_DIRECTORY
        / "scripts"
        / "search_arxiv.py"
    )

    if not script_path.exists():
        return (
            "Error: The arXiv search script was not found at "
            f"{script_path}"
        )

    command = [
        "uv",
        "run",
        "scripts/search_arxiv.py",
        "--query",
        query,
        "--max_results",
        str(max_results),
    ]

    try:
        result = subprocess.run(
            command,
            cwd=ARXIV_SKILL_DIRECTORY,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=120,
            check=False,
        )

    except subprocess.TimeoutExpired:
        return (
            "Error: The arXiv search timed out after "
            "120 seconds."
        )

    except OSError as error:
        return (
            "Error: The arXiv search process could not "
            f"be started: {error}"
        )

    if result.returncode != 0:
        error_message = (
            result.stderr.strip()
            or result.stdout.strip()
            or "Unknown command-line error."
        )

        return (
            "Error: The arXiv search command failed.\n"
            f"Details: {error_message}"
        )

    output = result.stdout.strip()

    if not output:
        return "Error: The arXiv search returned no output."

    return output

```

Query: thermodynamic simulation parameters for advanced fission reactors
Maximum results: 3

Searching arXiv...

=== ARXIV TOOL OUTPUT ===
{
  "status": "success",
  "results_count": 1,
  "papers": [
    {
      "id": "nucl-th/0602040v1",
      "title": "On The Double And Triple-Humped Fission Barriers And Half-Lives Of Actinide Elements",
      "pdf_url": "https://arxiv.org/pdf/nucl-th/0602040v1",
      "summary": "The potential energy of a deformed nucleus has been determined within a Generalized Liquid Drop Model taking into account the proximity energy, the microscopic corrections and compact and necked shapes. Multiple-humped potential barriers appear. A third minimum and third maximum exist in specific exit channels where one fragment is close to a magic spherical nucleus while the other one varies from oblate to prolate shapes. The heights of the fission barriers and half-lives of actinides are in agreement with the experimental results.",
      "published": "2006-02-13T13:11:24

### ADK Web UI verification

The following screenshot shows Scholar-Prime invoking the registered
`search_arxiv` tool, selecting the most relevant paper, summarizing its
abstract, and stating the available DOI.

![Scholar-Prime successfully invokes the arXiv search tool](../docs/assets/11E3.png)

## Exercise 4: Parameter Extraction & Verification

Define a structured parameter extraction tool. Write a python script that runs the full pipeline: searches arXiv, reads the top paper abstract, calls the extractor, and outputs a JSON file.

In [9]:
import inspect
import json
import subprocess

from scholar_prime.agent import (
    extract_parameters_from_text,
)


EXTRACTION_SCRIPT = (
    PROJECT_ROOT
    / "scripts"
    / "run_parameter_extraction.py"
)

OUTPUT_JSON = (
    PROJECT_ROOT
    / "data"
    / "simulation_parameters.json"
)


print("=== EXERCISE 4: PARAMETER EXTRACTION ===")

if not EXTRACTION_SCRIPT.exists():
    raise FileNotFoundError(
        f"Extraction script not found: {EXTRACTION_SCRIPT}"
    )

extractor_source = inspect.getsource(
    extract_parameters_from_text
)

display(
    Markdown(
        "### `extract_parameters_from_text` implementation\n\n"
        "```python\n"
        f"{extractor_source}\n"
        "```"
    )
)

execution_script_source = EXTRACTION_SCRIPT.read_text(
    encoding="utf-8"
)

display(
    Markdown(
        "### Complete extraction execution script\n\n"
        "```python\n"
        f"{execution_script_source}\n"
        "```"
    )
)

print("Running the complete extraction pipeline...\n")

execution_result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "scripts/run_parameter_extraction.py",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)

print(execution_result.stdout)

if execution_result.returncode != 0:
    print(execution_result.stderr)
    raise RuntimeError(
        "The parameter extraction pipeline failed."
    )

if not OUTPUT_JSON.exists():
    raise FileNotFoundError(
        f"Generated JSON file not found: {OUTPUT_JSON}"
    )

simulation_parameters = json.loads(
    OUTPUT_JSON.read_text(encoding="utf-8")
)

print("\n=== GENERATED simulation_parameters.json ===")

display(
    Markdown(
        "```json\n"
        + json.dumps(
            simulation_parameters,
            indent=2,
            ensure_ascii=False,
        )
        + "\n```"
    )
)

source_paper = simulation_parameters.get(
    "source_paper",
    {}
)

extraction = simulation_parameters.get(
    "extraction",
    {}
)

assert source_paper.get("doi")
assert extraction.get("status") == "success"
assert isinstance(
    extraction.get("parameters"),
    list,
)

print("✓ simulation_parameters.json was generated successfully.")
print(f"✓ Source DOI: {source_paper.get('doi')}")
print(
    "✓ Parameters found:",
    extraction.get("parameters_found"),
)

=== EXERCISE 4: PARAMETER EXTRACTION ===


### `extract_parameters_from_text` implementation

```python
def extract_parameters_from_text(text: str) -> dict[str, object]:
    """
    Extracts physical and simulation-related parameters from scientific text.

    The function searches for common reactor and material parameters, such as
    thermal conductivity, temperature, density, heat capacity, friction
    coefficients, fission barrier heights, and fission isomer energies.

    Args:
        text: Scientific abstract or paper text from which parameters
            should be extracted.

    Returns:
        A dictionary containing detected parameters, numerical values,
        units, evidence snippets, and an extraction summary.
    """

    if not text or not text.strip():
        return {
            "status": "error",
            "message": "The input text must not be empty.",
            "parameters": [],
        }

    parameter_definitions = [
        {
            "name": "thermal_conductivity",
            "label_pattern": r"thermal conductivity",
            "unit_pattern": r"W\s*/?\s*\(?m\s*[·*]?\s*K\)?",
            "default_unit": "W/(m·K)",
        },
        {
            "name": "temperature",
            "label_pattern": r"(?:temperature|core temperature)",
            "unit_pattern": r"(?:K|°C|Celsius|Kelvin)",
            "default_unit": None,
        },
        {
            "name": "density",
            "label_pattern": (
                r"\b(?:material density|mass density|bulk density|fuel density)\b"
            ),
            "unit_pattern": (
                r"(?:kg\s*/\s*m(?:\^?3|³)|g\s*/\s*cm(?:\^?3|³))"
            ),
            "default_unit": None,
        },
        {
            "name": "specific_heat_capacity",
            "label_pattern": r"(?:specific heat capacity|heat capacity)",
            "unit_pattern": r"J\s*/\s*\(?kg\s*[·*]?\s*K\)?",
            "default_unit": "J/(kg·K)",
        },
        {
            "name": "thermal_friction_coefficient",
            "label_pattern": r"(?:thermal friction coefficient|friction coefficient)",
            "unit_pattern": r"(?:dimensionless|1)",
            "default_unit": "dimensionless",
        },
        {
            "name": "fission_barrier_height",
            "label_pattern": r"(?:inner|outer|fission)?\s*barrier heights?",
            "unit_pattern": r"(?:MeV|keV|eV)",
            "default_unit": "MeV",
        },
        {
            "name": "fission_isomer_energy",
            "label_pattern": r"(?:fission )?isomer energies?",
            "unit_pattern": r"(?:MeV|keV|eV)",
            "default_unit": "MeV",
        },
    ]

    extracted_parameters: list[dict[str, object]] = []

    number_pattern = r"[-+]?\d+(?:[.,]\d+)?(?:[eE][-+]?\d+)?"

    for definition in parameter_definitions:
        label_pattern = definition["label_pattern"]
        unit_pattern = definition["unit_pattern"]

        mention_match = re.search(
            label_pattern,
            text,
            flags=re.IGNORECASE,
        )

        if mention_match is None:
            continue

        value_pattern = (
            rf"{label_pattern}"
            rf".{{0,100}}?"
            rf"(?P<value>{number_pattern})"
            rf"\s*(?P<unit>{unit_pattern})"
        )

        value_match = re.search(
            value_pattern,
            text,
            flags=re.IGNORECASE | re.DOTALL,
        )

        snippet_start = max(0, mention_match.start() - 60)
        snippet_end = min(len(text), mention_match.end() + 120)

        evidence = " ".join(
            text[snippet_start:snippet_end].split()
        )

        if value_match:
            raw_value = value_match.group("value").replace(",", ".")
            unit = value_match.group("unit")

            try:
                value: float | None = float(raw_value)
            except ValueError:
                value = None

            status = "numeric_value_extracted"
        else:
            value = None
            unit = definition["default_unit"]
            status = "mentioned_without_numeric_value"

        extracted_parameters.append(
            {
                "name": definition["name"],
                "value": value,
                "unit": unit,
                "status": status,
                "evidence": evidence,
            }
        )

    formulas = re.findall(
        r"\b[A-Za-z][A-Za-z0-9_]*\s*=\s*[^,.;\n]+",
        text,
    )

    return {
        "status": "success",
        "parameters_found": len(extracted_parameters),
        "parameters": extracted_parameters,
        "formulas": formulas,
        "note": (
            "Parameters without explicit numerical values are stored "
            "with value null instead of being invented."
        ),
    }

```

### Complete extraction execution script

```python
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any


# Hauptverzeichnis des Projekts bestimmen
PROJECT_ROOT = Path(__file__).resolve().parent.parent

# Sicherstellen, dass scholar_prime importiert werden kann,
# auch wenn dieses Skript aus dem scripts-Ordner gestartet wird.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from scholar_prime.agent import (  # noqa: E402
    extract_parameters_from_text,
    root_agent,
    search_arxiv,
)


SEARCH_QUERY = (
    "thermodynamic simulation parameters for advanced fission reactors"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "simulation_parameters.json"
)


def parse_json_objects(raw_output: str) -> list[dict[str, Any]]:
    """
    Extracts all valid JSON objects from the arXiv CLI output.

    The Science-Skills arXiv script may print several consecutive JSON
    objects while collecting results. This function parses every valid
    object so that the final and most complete result can be selected.

    Args:
        raw_output: Complete console output returned by search_arxiv.

    Returns:
        A list containing all successfully parsed JSON objects.

    Raises:
        ValueError: If no valid JSON object can be found.
    """

    decoder = json.JSONDecoder()
    parsed_objects: list[dict[str, Any]] = []
    position = 0

    while position < len(raw_output):
        object_start = raw_output.find("{", position)

        if object_start == -1:
            break

        try:
            parsed_object, object_end = decoder.raw_decode(
                raw_output,
                object_start,
            )
        except json.JSONDecodeError:
            position = object_start + 1
            continue

        if isinstance(parsed_object, dict):
            parsed_objects.append(parsed_object)

        position = object_end

    if not parsed_objects:
        raise ValueError(
            "The arXiv search output did not contain valid JSON."
        )

    return parsed_objects


def select_complete_search_result(
    search_results: list[dict[str, Any]],
) -> dict[str, Any]:
    """
    Selects the successful search result containing the most papers.

    Args:
        search_results: Parsed JSON objects returned by the arXiv CLI.

    Returns:
        The successful result with the largest papers list.

    Raises:
        ValueError: If no successful result containing papers exists.
    """

    successful_results = [
        result
        for result in search_results
        if result.get("status") == "success"
        and isinstance(result.get("papers"), list)
        and result["papers"]
    ]

    if not successful_results:
        raise ValueError(
            "The arXiv search did not return any papers."
        )

    return max(
        successful_results,
        key=lambda result: len(result["papers"]),
    )


def calculate_relevance_score(
    paper: dict[str, Any],
) -> int:
    """
    Calculates a simple relevance score for a scientific paper.

    Papers containing terms related to reactor simulation, fission,
    thermodynamics, material properties, and simulation parameters
    receive a higher score.

    Args:
        paper: Paper metadata containing a title and abstract.

    Returns:
        Integer relevance score.
    """

    title = str(paper.get("title", ""))
    abstract = str(paper.get("summary", ""))

    searchable_text = f"{title} {abstract}".lower()

    keyword_weights = {
        "reactor": 6,
        "thermodynamic": 6,
        "thermal": 5,
        "simulation": 5,
        "parameter": 4,
        "fission": 4,
        "energy density": 4,
        "barrier height": 4,
        "material": 3,
        "conductivity": 3,
        "temperature": 3,
        "dynamics": 2,
        "actinide": 2,
    }

    score = sum(
        weight
        for keyword, weight in keyword_weights.items()
        if keyword in searchable_text
    )

    # Ein Paper mit DOI ist für die spätere Verifikation besser geeignet.
    if paper.get("doi"):
        score += 2

    return score


def select_most_relevant_paper(
    papers: list[dict[str, Any]],
) -> dict[str, Any]:
    """
    Selects the paper with the highest calculated relevance score.

    Args:
        papers: Papers returned by the arXiv search.

    Returns:
        The paper considered most relevant for the search objective.

    Raises:
        ValueError: If the paper list is empty.
    """

    if not papers:
        raise ValueError("No papers are available for selection.")

    return max(
        papers,
        key=calculate_relevance_score,
    )


def main() -> None:
    """
    Runs the complete literature search and parameter extraction chain.
    """

    print("=== SCHOLAR-PRIME PARAMETER EXTRACTION ===")
    print(f"Agent: {root_agent.name}")
    print(f"Query: {SEARCH_QUERY}")

    print("\nSearching arXiv...")

    raw_search_output = search_arxiv(
        query=SEARCH_QUERY,
        max_results=3,
    )

    if raw_search_output.startswith("Error:"):
        raise RuntimeError(raw_search_output)

    parsed_results = parse_json_objects(raw_search_output)

    complete_result = select_complete_search_result(
        parsed_results
    )

    papers = complete_result["papers"]

    selected_paper = select_most_relevant_paper(papers)

    title = selected_paper.get("title")
    abstract = selected_paper.get("summary")
    doi = selected_paper.get("doi")

    if not abstract:
        raise ValueError(
            "The selected paper does not contain an abstract."
        )

    print("\nMost relevant paper:")
    print(f"Title: {title}")
    print(f"DOI: {doi or 'No DOI available'}")
    print(
        "Relevance score:",
        calculate_relevance_score(selected_paper),
    )

    print("\nExtracting simulation parameters...")

    extraction_result = extract_parameters_from_text(
        abstract
    )

    output_data = {
        "agent": root_agent.name,
        "search_query": SEARCH_QUERY,
        "source_paper": {
            "title": title,
            "authors": selected_paper.get("authors", []),
            "doi": doi,
            "arxiv_id": selected_paper.get("id"),
            "pdf_url": selected_paper.get("pdf_url"),
            "published": selected_paper.get("published"),
            "abstract": abstract,
        },
        "extraction": extraction_result,
    }

    OUTPUT_FILE.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    OUTPUT_FILE.write_text(
        json.dumps(
            output_data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("\nExtraction completed successfully.")
    print(f"Output file: {OUTPUT_FILE}")

    print("\n=== GENERATED JSON ===")
    print(
        json.dumps(
            output_data,
            indent=2,
            ensure_ascii=False,
        )
    )


if __name__ == "__main__":
    main()
```

Running the complete extraction pipeline...

=== SCHOLAR-PRIME PARAMETER EXTRACTION ===
Agent: scholar_prime
Query: thermodynamic simulation parameters for advanced fission reactors

Searching arXiv...

Most relevant paper:
Title: Microscopic Description of Nuclear Fission: Fission Barrier Heights of Even-Even Actinides
DOI: 10.1142/9789814525435_0072
Relevance score: 16

Extracting simulation parameters...

Extraction completed successfully.
Output file: C:\Users\nilss\Documents\genesis-oracle\data\simulation_parameters.json

=== GENERATED JSON ===
{
  "agent": "scholar_prime",
  "search_query": "thermodynamic simulation parameters for advanced fission reactors",
  "source_paper": {
    "title": "Microscopic Description of Nuclear Fission: Fission Barrier Heights of Even-Even Actinides",
    "authors": [
      "J. McDonnell",
      "N. Schunck",
      "W. Nazarewicz"
    ],
    "doi": "10.1142/9789814525435_0072",
    "arxiv_id": "1301.7587v1",
    "pdf_url": "https://arxiv.org/pdf/13

```json
{
  "agent": "scholar_prime",
  "search_query": "thermodynamic simulation parameters for advanced fission reactors",
  "source_paper": {
    "title": "Microscopic Description of Nuclear Fission: Fission Barrier Heights of Even-Even Actinides",
    "authors": [
      "J. McDonnell",
      "N. Schunck",
      "W. Nazarewicz"
    ],
    "doi": "10.1142/9789814525435_0072",
    "arxiv_id": "1301.7587v1",
    "pdf_url": "https://arxiv.org/pdf/1301.7587v1",
    "published": "2013-01-31T12:21:42Z",
    "abstract": "We evaluate the performance of modern nuclear energy density functionals for predicting inner and outer fission barrier heights and energies of fission isomers of even-even actinides. For isomer energies and outer barrier heights, we find that the self-consistent theory at the HFB level is capable of providing quantitative agreement with empirical data."
  },
  "extraction": {
    "status": "success",
    "parameters_found": 2,
    "parameters": [
      {
        "name": "fission_barrier_height",
        "value": null,
        "unit": "MeV",
        "status": "mentioned_without_numeric_value",
        "evidence": "r energy density functionals for predicting inner and outer fission barrier heights and energies of fission isomers of even-even actinides. For isomer energies and outer barrier heights, we find that the"
      },
      {
        "name": "fission_isomer_energy",
        "value": null,
        "unit": "MeV",
        "status": "mentioned_without_numeric_value",
        "evidence": "and energies of fission isomers of even-even actinides. For isomer energies and outer barrier heights, we find that the self-consistent theory at the HFB level is capable of providing quantitativ"
      }
    ],
    "formulas": [],
    "note": "Parameters without explicit numerical values are stored with value null instead of being invented."
  }
}
```

✓ simulation_parameters.json was generated successfully.
✓ Source DOI: 10.1142/9789814525435_0072
✓ Parameters found: 2


In [11]:
print("=== PROBLEM SET 11 FINAL VALIDATION ===")

required_files = {
    "Scholar-Prime agent": (
        PROJECT_ROOT
        / "scholar_prime"
        / "agent.py"
    ),
    "Extraction execution script": (
        PROJECT_ROOT
        / "scripts"
        / "run_parameter_extraction.py"
    ),
    "Generated simulation parameters": (
        PROJECT_ROOT
        / "data"
        / "simulation_parameters.json"
    ),
    "Exercise 3 screenshot": (
        PROJECT_ROOT
        / "docs"
        / "assets"
        / "11E3.png"
    ),
}

all_files_present = True

for description, file_path in required_files.items():
    exists = file_path.exists()
    status = "✓" if exists else "✗"

    print(
        f"{status} {description}: "
        f"{file_path.relative_to(PROJECT_ROOT)}"
    )

    if not exists:
        all_files_present = False

print("\nScience-Skills directories:")

science_skill_directories = [
    (
        PROJECT_ROOT
        / "science-skills"
        / "skills"
        / "literature_search_openalex"
    ),
    (
        PROJECT_ROOT
        / "science-skills"
        / "skills"
        / "literature_search_arxiv"
    ),
]

for directory in science_skill_directories:
    exists = directory.exists()
    status = "✓" if exists else "✗"

    print(
        f"{status} "
        f"{directory.relative_to(PROJECT_ROOT)}"
    )

    if not exists:
        all_files_present = False

if not all_files_present:
    raise FileNotFoundError(
        "At least one required submission file is missing."
    )

print("\n✓ All Problem Set 11 components are present.")

=== PROBLEM SET 11 FINAL VALIDATION ===
✓ Scholar-Prime agent: scholar_prime\agent.py
✓ Extraction execution script: scripts\run_parameter_extraction.py
✓ Generated simulation parameters: data\simulation_parameters.json
✓ Exercise 3 screenshot: docs\assets\11E3.png

Science-Skills directories:
✓ science-skills\skills\literature_search_openalex
✓ science-skills\skills\literature_search_arxiv

✓ All Problem Set 11 components are present.
